# Agente Aspirador — Table-Driven Agent

Agente reativo simples que consulta uma tabela percepção → ação no mundo do aspirador de dois cômodos. Sem busca e sem planejamento: serve de linha de base para comparar com os agentes seguintes.

**Técnica:** Agente reativo simples (table-driven)  
**Referência:** Russell & Norvig, *Inteligência Artificial* (AIMA)  
**Contexto:** disciplina de Inteligência Artificial — Análise e Desenvolvimento de Sistemas, FATEC Taubaté

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/devcauas/agentes-ia/blob/main/notebooks/01-agente-aspirador-tabela.ipynb)


In [1]:
# =============================================================================
# AGENTE ASPIRADOR - TABLE-DRIVEN AGENT
# Implementação clássica do Mundo do Aspirador (Russel & Norvig)
# =============================================================================

# -----------------------------------------------------------------------------
# FUNDAMENTAÇÃO TEÓRICA
#
# Agente Reativo Simples baseado em Tabela (Table-Driven Agent):
#   - Percebe o estado atual do ambiente via sensores
#   - Consulta uma tabela de mapeamento estado → ação
#   - Executa a ação correspondente via atuadores
#   - NÃO planeja, NÃO busca, NÃO usa heurísticas
#
# O estado é definido por 3 componentes:
#   1. Posição do robô: "A" ou "B"
#   2. Situação de A:   "Sujo" ou "Limpo"
#   3. Situação de B:   "Sujo" ou "Limpo"
#
# Total de estados: 2 posições × 2² combinações de limpeza = 8 estados
#
# Ações disponíveis: ASPIRAR | ESQUERDA | DIREITA
# Custo por ação: 1 (custo uniforme)
# Objetivo: A limpo E B limpo (independente da posição do robô)
# -----------------------------------------------------------------------------


# =============================================================================
# CLASSE: Ambiente
# Representa o mundo do aspirador — discreto, totalmente observável,
# determinístico e estático (exceto pelas ações do agente).
# =============================================================================

class Ambiente:
    def __init__(self, posicao_inicial: str, status_a: str, status_b: str):
        """
        Inicializa o ambiente com um estado específico.

        Parâmetros:
            posicao_inicial : "A" ou "B" — onde o robô começa
            status_a        : "Sujo" ou "Limpo" — estado da posição A
            status_b        : "Sujo" ou "Limpo" — estado da posição B

        O estado é representado como uma tupla imutável:
            (posição, status_A, status_B)
        Exemplo: ("A", "Sujo", "Limpo")
        Essa representação é compacta, hashável e diretamente usável
        como chave no dicionário da tabela de decisões.
        """
        self.posicao = posicao_inicial
        self.status_a = status_a
        self.status_b = status_b

    def get_estado(self) -> tuple:
        """
        Sensor: retorna a percepção completa do ambiente como tupla.
        Em um ambiente totalmente observável, o agente enxerga tudo.

        Retorna:
            tuple: (posição_robô, status_A, status_B)
        """
        return (self.posicao, self.status_a, self.status_b)

    def aplicar_acao(self, acao: str) -> tuple:
        """
        Modelo de Transição: dado o estado atual e uma ação,
        retorna o novo estado resultante.

        Transições definidas:
            ASPIRAR   → limpa a posição atual do robô
            ESQUERDA  → move robô para A (sem efeito se já estiver em A)
            DIREITA   → move robô para B (sem efeito se já estiver em B)

        Ações podem ser "sem efeito" (self-loops no espaço de estados).
        Custo = 1 para qualquer ação executada.

        Parâmetros:
            acao : str — "ASPIRAR", "ESQUERDA" ou "DIREITA"

        Retorna:
            tuple: novo estado após a transição
        """
        if acao == "ASPIRAR":
            # Remove sujeira da posição atual do robô
            if self.posicao == "A":
                self.status_a = "Limpo"
            else:
                self.status_b = "Limpo"

        elif acao == "ESQUERDA":
            # Move para A; se já estiver em A, não altera nada
            self.posicao = "A"

        elif acao == "DIREITA":
            # Move para B; se já estiver em B, não altera nada
            self.posicao = "B"

        return self.get_estado()

    def objetivo_alcancado(self) -> bool:
        """
        Teste de Objetivo: verifica se ambas as posições estão limpas.
        A posição do robô não importa para o objetivo.

        Retorna:
            bool: True se A e B estiverem limpos
        """
        return self.status_a == "Limpo" and self.status_b == "Limpo"

    def exibir_estado(self):
        """Exibe o estado atual do ambiente de forma formatada."""
        print(f"  Robô : {self.posicao}")
        print(f"  A    : {self.status_a}")
        print(f"  B    : {self.status_b}")


# =============================================================================
# TABELA DE DECISÕES
#
# Coração do Table-Driven Agent.
# Mapeia DIRETAMENTE cada estado possível à ação correta.
# Não há lógica condicional complexa — apenas consulta O(1).
#
# Os 8 estados possíveis e suas ações ótimas:
#
# Estado              | Raciocínio                        | Ação
# --------------------|-----------------------------------|----------
# (A, Sujo,  Sujo)   | Robô em A, A sujo → aspira aqui  | ASPIRAR
# (A, Sujo,  Limpo)  | Robô em A, A sujo → aspira aqui  | ASPIRAR
# (A, Limpo, Sujo)   | A limpo, B sujo → vai para B      | DIREITA
# (A, Limpo, Limpo)  | Tudo limpo → objetivo alcançado   | PARAR *
# (B, Sujo,  Sujo)   | Robô em B, B sujo → aspira aqui  | ASPIRAR
# (B, Sujo,  Limpo)  | B sujo → aspira aqui              | ASPIRAR
# (B, Limpo, Sujo)   | B limpo, A sujo → vai para A      | ESQUERDA
# (B, Limpo, Limpo)  | Tudo limpo → objetivo alcançado   | PARAR *
#
# * PARAR não é uma ação do agente — o loop principal detecta o objetivo.
# =============================================================================

TABELA_DECISOES = {
    # --- Robô em A ---
    ("A", "Sujo",  "Sujo") : "ASPIRAR",   # A sujo: aspira A primeiro
    ("A", "Sujo",  "Limpo"): "ASPIRAR",   # A sujo: aspira A
    ("A", "Limpo", "Sujo") : "DIREITA",   # A limpo, B sujo: move para B
    ("A", "Limpo", "Limpo"): None,        # Objetivo! Nenhuma ação necessária

    # --- Robô em B ---
    ("B", "Sujo",  "Sujo") : "ASPIRAR",   # B sujo: aspira B primeiro
    ("B", "Sujo",  "Limpo"): "ESQUERDA",  # B limpo, A sujo: move para A (*)
    ("B", "Limpo", "Sujo") : "ASPIRAR",   # B sujo: aspira B
    ("B", "Limpo", "Limpo"): None,        # Objetivo! Nenhuma ação necessária
}

# (*) Nota: ("B","Sujo","Limpo") — robô em B, A está sujo, B está limpo.
#     A ação correta é ESQUERDA para ir limpar A.


# =============================================================================
# CLASSE: Agente
# Implementa o Table-Driven Agent (Russel & Norvig, Cap. 2).
#
# Ciclo de vida do agente:
#   1. Perceber → lê sensores do ambiente
#   2. Consultar → busca ação na tabela
#   3. Agir     → executa a ação no ambiente
# =============================================================================

class Agente:
    def __init__(self, tabela: dict):
        """
        Inicializa o agente com sua tabela de decisões.

        O agente não possui memória interna de estados anteriores.
        Cada decisão é tomada com base APENAS na percepção atual.
        Isso o caracteriza como um agente reativo simples.

        Parâmetros:
            tabela : dict — mapeamento {estado → ação}
        """
        self.tabela = tabela
        self.custo_acumulado = 0
        self.historico_acoes = []

    def perceber(self, ambiente: Ambiente) -> tuple:
        """
        Função de percepção: lê o estado atual do ambiente.
        Em ambientes totalmente observáveis, percepção = estado completo.

        Retorna:
            tuple: estado atual (posição, status_A, status_B)
        """
        return ambiente.get_estado()

    def decidir(self, estado: tuple) -> str:
        """
        Função de decisão: consulta a tabela para o estado dado.
        Esta é a essência do Table-Driven Agent — sem cálculo, sem busca.

        Parâmetros:
            estado : tuple — percepção atual

        Retorna:
            str ou None: ação a executar, ou None se objetivo alcançado
        """
        return self.tabela.get(estado, None)

    def agir(self, acao: str, ambiente: Ambiente) -> tuple:
        """
        Função de ação: aplica a ação no ambiente (atuador).
        Registra a ação no histórico e incrementa o custo.

        Custo uniforme: cada ação custa 1 unidade.

        Parâmetros:
            acao     : str — ação a executar
            ambiente : Ambiente — mundo onde a ação é aplicada

        Retorna:
            tuple: novo estado após a ação
        """
        self.historico_acoes.append(acao)
        self.custo_acumulado += 1  # Custo uniforme: c(ação) = 1
        return ambiente.aplicar_acao(acao)

    def exibir_historico(self):
        """Exibe o resumo completo da execução do agente."""
        print("\n" + "=" * 55)
        print("  RESUMO DA EXECUÇÃO")
        print("=" * 55)
        print(f"  Sequência de ações : {' → '.join(self.historico_acoes)}")
        print(f"  Total de ações     : {len(self.historico_acoes)}")
        print(f"  Custo acumulado    : {self.custo_acumulado}")
        print("=" * 55)


# =============================================================================
# ESPAÇO DE ESTADOS
#
# Função auxiliar para exibir todos os estados possíveis do ambiente.
# Serve como material didático para compreender a estrutura do problema.
# =============================================================================

def exibir_espaco_de_estados():
    """
    Exibe e explica os 8 estados possíveis do Mundo do Aspirador.

    Espaço de Estados ≠ Árvore de Busca:
      - Espaço de estados: conjunto de todos os estados alcançáveis
        a partir do estado inicial via ações. É um grafo.
      - Árvore de busca: estrutura gerada ao explorar o espaço de estados
        sistematicamente (pode conter estados repetidos).

    O agente de tabela não constrói nem percorre a árvore de busca.
    Ele simplesmente consulta o mapeamento pré-definido.
    """
    print("\n" + "=" * 55)
    print("  ESPAÇO DE ESTADOS DO MUNDO DO ASPIRADOR")
    print("=" * 55)
    print("  Fórmula: 2 posições × 2² status = 8 estados\n")

    estados = [
        ("A", "Sujo",  "Sujo",  "← Estado inicial típico"),
        ("A", "Sujo",  "Limpo", ""),
        ("A", "Limpo", "Sujo",  ""),
        ("A", "Limpo", "Limpo", "← OBJETIVO"),
        ("B", "Sujo",  "Sujo",  ""),
        ("B", "Sujo",  "Limpo", ""),
        ("B", "Limpo", "Sujo",  ""),
        ("B", "Limpo", "Limpo", "← OBJETIVO"),
    ]

    for i, (pos, sa, sb, obs) in enumerate(estados, 1):
        acao = TABELA_DECISOES.get((pos, sa, sb), "—")
        acao_str = acao if acao else "PARAR"
        print(f"  {i}. ({pos}, A={sa:5s}, B={sb:5s}) → {acao_str:10s} {obs}")

    print("\n  Ambiente: discreto | totalmente observável |")
    print("            determinístico | estático | pequeno")
    print("  → Ideal para agentes reativos simples baseados em tabela")
    print("=" * 55)


# =============================================================================
# SIMULAÇÃO PRINCIPAL
# Loop de percepção → decisão → ação até atingir o objetivo.
# =============================================================================

def executar_simulacao(posicao_inicial: str, status_a: str, status_b: str,
                        max_passos: int = 10):
    """
    Executa a simulação completa do Agente Aspirador.

    Parâmetros:
        posicao_inicial : "A" ou "B"
        status_a        : "Sujo" ou "Limpo"
        status_b        : "Sujo" ou "Limpo"
        max_passos      : limite de segurança para evitar loops infinitos
    """
    print("\n" + "=" * 55)
    print("  AGENTE ASPIRADOR — TABLE-DRIVEN AGENT")
    print("  Mundo do Aspirador | Agente Reativo Simples")
    print("=" * 55)

    # Instancia o ambiente e o agente
    ambiente = Ambiente(posicao_inicial, status_a, status_b)
    agente = Agente(TABELA_DECISOES)

    print("\n  ESTADO INICIAL:")
    ambiente.exibir_estado()

    # Verifica se o estado inicial já é o objetivo
    if ambiente.objetivo_alcancado():
        print("\n  Ambiente já estava limpo. Nenhuma ação necessária.")
        agente.exibir_historico()
        return

    print()

    # ------------------------------------------------------------------
    # LOOP PRINCIPAL DO AGENTE
    # Implementa o ciclo: perceber → decidir → agir
    # Termina quando o objetivo é alcançado ou o limite é atingido
    # ------------------------------------------------------------------
    for passo in range(1, max_passos + 1):

        print(f"  {'─' * 51}")
        print(f"  PASSO {passo}")
        print(f"  {'─' * 51}")

        # 1. PERCEBER — lê os sensores do ambiente
        estado_atual = agente.perceber(ambiente)
        print(f"  Estado percebido : {estado_atual}")

        # 2. DECIDIR — consulta a tabela de decisões
        acao = agente.decidir(estado_atual)

        if acao is None:
            # Estado mapeado para None → objetivo alcançado
            print("  Ação             : PARAR (objetivo detectado)")
            break

        print(f"  Ação escolhida   : {acao}")

        # 3. AGIR — executa a ação e atualiza o ambiente
        estado_anterior = estado_atual
        novo_estado = agente.agir(acao, ambiente)

        print(f"  Transição        : {estado_anterior} → {novo_estado}")
        print(f"  Custo acumulado  : {agente.custo_acumulado}")

        # Verifica se o objetivo foi alcançado após a ação
        if ambiente.objetivo_alcancado():
            print()
            print("  ✔ OBJETIVO ALCANÇADO!")
            print("    A: Limpo | B: Limpo")
            break
    else:
        print("\n  ✘ Limite de passos atingido sem alcançar o objetivo.")

    # Exibe o estado final e o histórico
    print()
    print("  ESTADO FINAL:")
    ambiente.exibir_estado()
    agente.exibir_historico()


# =============================================================================
# PONTO DE ENTRADA
# =============================================================================

if __name__ == "__main__":

    # Exibe o espaço de estados completo (material didático)
    exibir_espaco_de_estados()

    # ------------------------------------------------------------------
    # CENÁRIOS DE TESTE
    # Cada cenário demonstra um caminho diferente no espaço de estados
    # ------------------------------------------------------------------

    cenarios = [
        {
            "descricao" : "Cenário 1 — Robô em A, ambos sujos (pior caso)",
            "posicao"   : "A",
            "status_a"  : "Sujo",
            "status_b"  : "Sujo",
        },
        {
            "descricao" : "Cenário 2 — Robô em A, só B sujo",
            "posicao"   : "A",
            "status_a"  : "Limpo",
            "status_b"  : "Sujo",
        },
        {
            "descricao" : "Cenário 3 — Robô em B, ambos sujos",
            "posicao"   : "B",
            "status_a"  : "Sujo",
            "status_b"  : "Sujo",
        },
        {
            "descricao" : "Cenário 4 — Ambiente já limpo (melhor caso)",
            "posicao"   : "A",
            "status_a"  : "Limpo",
            "status_b"  : "Limpo",
        },
    ]

    for cenario in cenarios:
        print(f"\n\n{'#' * 55}")
        print(f"  {cenario['descricao']}")
        print(f"{'#' * 55}")
        executar_simulacao(
            posicao_inicial = cenario["posicao"],
            status_a        = cenario["status_a"],
            status_b        = cenario["status_b"],
        )


  ESPAÇO DE ESTADOS DO MUNDO DO ASPIRADOR
  Fórmula: 2 posições × 2² status = 8 estados

  1. (A, A=Sujo , B=Sujo ) → ASPIRAR    ← Estado inicial típico
  2. (A, A=Sujo , B=Limpo) → ASPIRAR    
  3. (A, A=Limpo, B=Sujo ) → DIREITA    
  4. (A, A=Limpo, B=Limpo) → PARAR      ← OBJETIVO
  5. (B, A=Sujo , B=Sujo ) → ASPIRAR    
  6. (B, A=Sujo , B=Limpo) → ESQUERDA   
  7. (B, A=Limpo, B=Sujo ) → ASPIRAR    
  8. (B, A=Limpo, B=Limpo) → PARAR      ← OBJETIVO

  Ambiente: discreto | totalmente observável |
            determinístico | estático | pequeno
  → Ideal para agentes reativos simples baseados em tabela


#######################################################
  Cenário 1 — Robô em A, ambos sujos (pior caso)
#######################################################

  AGENTE ASPIRADOR — TABLE-DRIVEN AGENT
  Mundo do Aspirador | Agente Reativo Simples

  ESTADO INICIAL:
  Robô : A
  A    : Sujo
  B    : Sujo

  ───────────────────────────────────────────────────
  PASSO 1
  ─────────